# Phase 8b — Promotion TWFE Robustness Audit (heterogeneity / negative-weights)

**คำถาม:** ตัวเลข **+28.4% uplift** ใน Phase 8 มาจาก two-way fixed-effects regression บน `promotion_flag`
ซึ่งเป็น **repeating on/off treatment** (median 53.5 ครั้งสลับต่อ series ตลอด <=150 สัปดาห์ ไม่ใช่ staggered
adoption ทางเดียว) — de Chaisemartin & D'Haultfoeuille (2020) แสดงว่า TWFE มาตรฐานบน non-absorbing/switching
treatment แบบนี้เสี่ยงได้ **negative weights** จาก treatment-effect heterogeneity ทำให้สัมประสิทธิ์ที่ได้อาจ
ไม่ตรงกับค่าเฉลี่ยผลกระทบจริงของกลุ่มที่ได้รับ treatment (ดู `LITERATURE_GROUNDING.md` §3a สำหรับรายละเอียด
งานวิจัย)

**สิ่งที่ notebook นี้ทำ:**
1. สร้าง panel ที่ fixed-effect unit ละเอียดกว่า Phase 8 (sku x channel x region = 270 หน่วย แทนที่จะเป็น sku
   เดียว 30 หน่วย เพราะ `promotion_flag` แปรผันระดับ sku x channel x region ไม่ใช่แค่ sku) แล้ว refit TWFE
   เดิมบน grouping นี้เป็น **parity baseline** ก่อน — เพื่อยืนยันว่าความละเอียดของ FE ที่เปลี่ยนไปเองไม่ได้ทำให้
   ตัวเลขขยับ ก่อนจะเอาผลต่างจากเครื่องมือ heterogeneity-robust มาตีความ
2. รันเครื่องมือ heterogeneity-robust ที่แนะนำใน literature review จริง 3 ตัว: Python `did-multiplegt-stat`
   (static WAS/`DID_M` estimator), Python `py-did-multiplegt-dyn` (dynamic estimator สำหรับ pull-forward),
   และ R `TwoWayFEWeights` (negative-weights diagnostic) — บันทึกผลจริงที่เกิดขึ้น ไม่ว่าจะสำเร็จหรือไม่

**สรุปล่วงหน้า (ดูรายละเอียดด้านล่าง):** ทั้ง 3 เครื่องมือ error บน panel จริงของโปรเจกต์นี้ ด้วยสาเหตุภายในที่
ต่างกัน 3 แบบ — ข้อค้นพบนี้เองมีความหมาย: ไม่ใช่ implementation เดียวที่มีบั๊ก แต่เป็นรูปแบบการสลับ
promotion_flag ที่ถี่มาก (median ~47% โอกาสสลับต่อหน่วยต่อสัปดาห์) ที่ทำให้ internal logic ของทั้ง 3 เครื่องมือ
(ซึ่งมาจาก research group เดียวกัน) หาช่วงเปรียบเทียบ (comparison cells) ไม่ได้ **สถานะ: ยังไม่ได้แก้ — ตัวเลข
+28.4% ยังคงเป็นแค่ static TWFE estimate ที่ยังไม่ผ่านการตรวจ heterogeneity/negative-weights อย่างที่ตั้งใจ**


In [1]:
import sys
sys.path.insert(0, '../src')

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd

from causal.promotion_robustness import build_group_panel, fit_parity_twfe, export_group_panel_csv

pd.set_option('display.width', 160)

df = pd.read_csv('../data/processed/weekly_features.csv', parse_dates=['week'])
panel = build_group_panel(df)
print('panel shape:', panel.shape)
print('n groups (sku x channel x region):', panel.group_id.nunique())
print('n time periods:', panel.time_id.nunique())


panel shape: (31027, 43)
n groups (sku x channel x region): 270
n time periods: 150


## 1. Parity TWFE — group_id (sku x channel x region) แทน sku

ยืนยันก่อนว่าความละเอียดของ fixed effects ที่ต้องเปลี่ยนไปเพื่อให้เครื่องมือ heterogeneity-robust ทำงานได้ ไม่ได้ทำให้ตัวเลขเปลี่ยนไปเองจากที่ตีพิมพ์ใน Phase 8 (sku-level FE)

In [2]:
result = fit_parity_twfe(panel)
print(result.summary)

coef = result.params['promotion_flag']
lo, hi = result.conf_int().loc['promotion_flag']
uplift_pct = (np.expm1(coef)) * 100
print()
print(f'uplift_pct (group_id FE): {uplift_pct:.2f}%  (Phase 8 sku-level FE published: 28.41%)')
print(f'95% CI: [{(np.expm1(lo))*100:.2f}%, {(np.expm1(hi))*100:.2f}%]')
print(f'p-value: {result.pvalues["promotion_flag"]:.4g}')


                          PanelOLS Estimation Summary                           
Dep. Variable:              log_units   R-squared:                        0.1425
Estimator:                   PanelOLS   R-squared (Between):              0.0568
No. Observations:               31027   R-squared (Within):               0.1209
Date:                Sat, Sep 12 2026   R-squared (Overall):              0.0572
Time:                        02:05:25   Log-likelihood                   -6320.2
Cov. Estimator:             Clustered                                           
                                        F-statistic:                      2542.1
Entities:                         270   P-value                           0.0000
Avg Obs:                       114.91   Distribution:                 F(2,30606)
Min Obs:                       80.000                                           
Max Obs:                       150.00   F-statistic (robust):             3205.4
                            

**ผล:** ตัวเลข uplift ที่ group_id-level FE (sku x channel x region, 270 หน่วย) ตรงกับตัวเลข sku-level FE เดิม
ของ Phase 8 (28.41% ทั้งคู่ ต่างกันแค่ทศนิยมที่ 2) — ยืนยันว่า parity baseline ใช้ได้ ความแตกต่างใดๆ ที่จะเจอจาก
เครื่องมือ heterogeneity-robust ด้านล่างจึงเป็นผลจากตัว robustness check เอง ไม่ใช่ artifact จากการเปลี่ยน FE
granularity

## 2. Static heterogeneity-robust estimate — Python `did-multiplegt-stat`

`DID_M`/WAS estimator (ไม่ใช่ "Wald-TC" — คำนั้นใช้กับ fuzzy/IV design ซึ่งไม่ตรงกับ `promotion_flag` ที่เป็น
observed binary treatment ตรงๆ — ดู `LITERATURE_GROUNDING.md` §3a) เทียบกับตัวเลข TWFE ด้านบน

In [3]:
from did_multiplegt_stat import DIDMultiplegtStat

try:
    est = DIDMultiplegtStat(estimator=['was'], exact_match=True, switchers=None, cluster='sku', placebo=0)
    est.fit(panel, Y='log_units', ID='group_id', Time='time_id', D='promotion_flag')
    est.summary()
except Exception as e:
    print(f'FAILED: {type(e).__name__}: {e}')


FAILED: ZeroDivisionError: Weights sum to zero, can't be normalized


**ผล:** error ภายใน package เอง (`ZeroDivisionError`, weights sum to zero ตอน normalize) เกิดที่ internal
weight-computation step ไม่ใช่จาก input ที่ผิด — panel ผ่านทุก precondition ที่เช็คได้แล้ว (ไม่มี duplicate
group x time, treatment เป็น binary, ไม่มี gap ภายใน, มีทั้ง switcher และ stayer ในเกือบทุก period) ลองแล้วทั้ง
`exact_match=True/False`, `placebo` หลายค่า, `switchers=None/'up'/'down'` ผลเหมือนกันทุกครั้ง

## 3. Dynamic estimate (pull-forward) — Python `py-did-multiplegt-dyn`

In [4]:
import polars as pl
from did_multiplegt_dyn import DidMultiplegtDyn

pdf = pl.from_pandas(panel[['group_id', 'time_id', 'log_units', 'promotion_flag', 'sku']])

try:
    est = DidMultiplegtDyn(
        df=pdf, outcome='log_units', group='group_id', time='time_id',
        treatment='promotion_flag', effects=4, placebo=0, cluster='sku',
    )
    est.fit()
    est.summary()
except Exception as e:
    print(f'FAILED: {type(e).__name__}: {e}')


FAILED: TypeError: unsupported operand type(s) for +=: 'float' and 'NoneType'


**ผล:** error ภายใน package เช่นกัน (`TypeError`, `None` เข้าไปบวกกับ `float` ตอนคำนวณ placebo mask) —
เกิดขึ้นแม้ตั้ง `placebo=0` ซึ่งไม่ควรต้องรันโค้ดส่วน placebo เลยด้วยซ้ำ ลองแล้วหลาย config
(`effects`/`placebo` ต่างๆ, `same_switchers=True/False`) ผลเหมือนกันทุกครั้ง

## 4. Negative-weights diagnostic — R `TwoWayFEWeights`

Python port ของ `TwoWayFEWeights` (`py_twowayfeweights`) ยังใช้งานไม่ได้จริง (repo มีแค่ placeholder README)
จึงต้องรันผ่าน R — environment นี้ไม่มี R ติดตั้งไว้ จึงรันผ่าน disposable Docker container แทน
(`analysis/r/run_twfe_weights_docker.sh` + `analysis/r/promotion_twfe_weights.R`) ไม่ได้รันเป็น cell ในสมุด
บันทึกนี้โดยตรง (ข้าม kernel ไป R) — คำสั่งที่รันจริงและผลที่ได้:

```
$ python -c "from causal.promotion_robustness import build_group_panel, export_group_panel_csv; \
    import pandas as pd; df = pd.read_csv('data/processed/weekly_features.csv', parse_dates=['week']); \
    export_group_panel_csv(build_group_panel(df), 'reports/promotion_panel_export.csv')"
$ MSYS_NO_PATHCONV=1 bash analysis/r/run_twfe_weights_docker.sh
```

**ผลจริงที่ได้ (คัดมาจาก output จริง):**

```
[1] "The treatment variable in the regression varies within some group * period cells."
[1] "The results in de Chaisemartin, C. and D'Haultfoeuille, X. (2020) apply to two-way fixed effects regressions"
[1] "with a group * period level treatment."
[1] "The command will replace the treatment by its average value in each group * period."
Error: in fixest::feols(fml, data = dt, weights = dt$weight...:
The data set contains 0 observation: the estimation cannot be done.
Execution halted
```

คำเตือน "varies within some group * period cells" ขึ้นแม้ panel นี้จะมีแค่ 1 แถวต่อ (group_id, time_id) เป๊ะๆ
จริงๆ (ตรวจแล้วใน `build_group_panel()`) แสดงว่า step ภายในที่ collapse/average ค่า treatment เป็นตัวทำให้
working dataset ว่างเปล่า ไม่ใช่ปัญหา duplicate row จริง

## 5. สรุป

**เครื่องมือ heterogeneity-robust ทั้ง 3 ตัว (มาจาก research ecosystem เดียวกันของ de Chaisemartin &
D'Haultfoeuille แต่คนละ implementation — R หนึ่ง, Python สอง) error บน panel จริงของโปรเจกต์นี้ทั้งหมด ด้วยกลไก
ภายในที่ต่างกัน 3 แบบ** — ความสอดคล้องกันนี้เองบ่งชี้ว่าไม่ใช่บั๊กของ implementation ใดตัวหนึ่ง แต่เป็นรูปแบบการ
สลับ `promotion_flag` ที่ถี่มากในข้อมูลชุดนี้ (median ~47% โอกาสสลับต่อหน่วยต่อสัปดาห์ — ใกล้เคียงสุ่มทุกสัปดาห์)
ที่ทำให้ internal logic ของทั้ง 3 เครื่องมือ (ซึ่งออกแบบมาเพื่อหา "stayer"/"control" cells มาเทียบ) หาช่วง
เปรียบเทียบที่ใช้ได้ไม่เจอ

**สถานะ:** ยังไม่ได้แก้ ณ ตอนที่เขียน revision นี้ — ทางเลือกที่เหลือคือ (1) รอ/รายงานบั๊กไปยังผู้พัฒนา package,
(2) เขียน decomposition เองจากสูตรในเปเปอร์ต้นฉบับ (ความเสี่ยงสูงที่จะผิดโดยไม่มี reference implementation
มาเทียบ, ไม่ได้ลองในรอบนี้), หรือ (3) ยอมรับว่า diagnostic นี้คำนวณไม่ได้ด้วยเครื่องมือสำเร็จรูปสำหรับ panel
รูปแบบนี้โดยเฉพาะ และรายงานข้อจำกัดนี้ตรงๆ (ทางเลือกที่ใช้ในตอนนี้) **ตัวเลข +28.4% ใน Phase 8 จึงยังคงเป็นแค่
static TWFE estimate ที่ยังไม่ผ่านการตรวจ heterogeneity/negative-weights bias อย่างที่ตั้งใจไว้** (ดู README,
`08_promotion_effect.ipynb` ข้อ 4, และ `LITERATURE_GROUNDING.md` §3a สำหรับ framing เดียวกัน)

Pull-forward (การทดสอบว่าโปรโมชันแค่ดึงยอดขายในอนาคตมาขายก่อนหรือเปล่า) ขึ้นอยู่กับเครื่องมือ dynamic estimator
ตัวเดียวกัน (ข้อ 3) จึงยังเป็นงานเปิดเช่นกัน ด้วยเหตุผลเดียวกัน
